In [0]:
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.feature import Imputer, StringIndexer, OneHotEncoder
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.functions import vector_to_array

from pyspark.sql.window import Window

train = spark.table("workspace.mortgage_risk.model_train")
validation = spark.table("workspace.mortgage_risk.model_validation")

numeric_features = [
    "fico",
    "ltv",
    "additional_lien_ltv",
    "dti",
    "original_upb",
    "original_rate",
    "original_term",
    "mi_pct",
    "multiple_borrowers",
    "units",
    "original_age_months",
    "loan_age",
    "remaining_term",
    "current_rate",
    "rate_change",
    "estimated_ltv",
    "balance_ratio",
    "delinquency_months",
    "late_months_3m",
    "late_months_6m",
    "max_delinquency_3m",
    "max_delinquency_6m"
]

categorical_features = [
    "purpose",
    "occupancy",
    "property_type",
    "state",
    "channel"
]

In [0]:
# Build the preprocessing pipeline
imputed_columns = []

for column in numeric_features:
    imputed_columns.append(column + "_imputed")

imputer = Imputer(
    strategy="median",
    inputCols=numeric_features,
    outputCols=imputed_columns
)

stages = [imputer]
encoded_columns = []

for column in categorical_features:
    index_column = column + "_index"
    encoded_column = column + "_encoded"

    indexer = StringIndexer(
        inputCol=column,
        outputCol=index_column,
        handleInvalid="keep"
    )

    encoder = OneHotEncoder(
        inputCol=index_column,
        outputCol=encoded_column,
        handleInvalid="keep",
        dropLast=True
    )

    stages.append(indexer)
    stages.append(encoder)
    encoded_columns.append(encoded_column)

assembler = VectorAssembler(
    inputCols=imputed_columns + encoded_columns,
    outputCol="features"
)

stages.append(assembler)

In [0]:
# Add logistic regression and fit on training data
logistic_regression = LogisticRegression(
    featuresCol="features",
    labelCol="label_12m",
    maxIter=100,
    regParam=0.01,
    elasticNetParam=0.0,
    standardization=True
)

stages.append(logistic_regression)

pipeline = Pipeline(stages=stages)

baseline_model = pipeline.fit(train)

print("Logistic regression baseline fitted.")

Logistic regression baseline fitted.


In [0]:
# Score validation and calculate the first metrics
validation_predictions = baseline_model.transform(validation)

validation_scores = validation_predictions.withColumn(
    "risk_score",
    vector_to_array("probability")[1]
).select(
    "loan_id",
    "prediction_month",
    "label_12m",
    "delinquency_months",
    "risk_score"
)

pr_evaluator = BinaryClassificationEvaluator(
    labelCol="label_12m",
    rawPredictionCol="risk_score",
    metricName="areaUnderPR",
    numBins=10000
)

pr_auc = pr_evaluator.evaluate(validation_scores)

print(f"Validation PR-AUC: {pr_auc:.6f}")

calibration_summary = validation_scores.agg(
    F.avg("risk_score").alias("average_predicted_risk"),
    F.avg("label_12m").alias("observed_event_rate"),
    F.avg(
        F.pow(F.col("risk_score") - F.col("label_12m"), 2)
    ).alias("brier_score")
)

display(calibration_summary)

Validation PR-AUC: 0.147474


average_predicted_risk,observed_event_rate,brier_score
0.018023464847820133,0.004917968383572671,0.0049761004080809456


In [0]:
# Count monthly alerts and captured events
def monthly_alert_counts(scores):
    month_window = Window.partitionBy("prediction_month")

    # Use a reproducible tie-breaker when ranking scores are equal.
    rank_window = month_window.orderBy(
        F.col("ranking_score").desc(),
        F.xxhash64("loan_id", "prediction_month"),
        F.col("loan_id")
    )

    ranked = scores.withColumn(
        "monthly_loans",
        F.count("*").over(month_window)
    )

    ranked = ranked.withColumn(
        "risk_rank",
        F.row_number().over(rank_window)
    )

    ranked = ranked.withColumn(
        "alert_limit",
        F.ceil(F.col("monthly_loans") * 0.05)
    )

    ranked = ranked.withColumn(
        "alert",
        F.when(
            F.col("risk_rank") <= F.col("alert_limit"), 1
        ).otherwise(0)
    )

    ranked = ranked.withColumn(
        "event_caught",
        F.when(
            F.col("alert") == 1,
            F.col("label_12m")
        ).otherwise(0)
    )

    monthly_counts = ranked.groupBy("prediction_month").agg(
        F.count("*").alias("loans"),
        F.sum("label_12m").alias("events"),
        F.sum("alert").alias("alerts"),
        F.sum("event_caught").alias("events_caught")
    )

    return monthly_counts

In [0]:
def summarize_alerts(monthly_counts, model_name, population):
    summary = monthly_counts.agg(
        F.sum("loans").alias("loans"),
        F.sum("events").alias("events"),
        F.sum("alerts").alias("alerts"),
        F.sum("events_caught").alias("events_caught")
    )

    summary = summary.withColumn(
        "precision",
        F.when(
            F.col("alerts") > 0,
            F.col("events_caught") / F.col("alerts")
        )
    )

    summary = summary.withColumn(
        "recall",
        F.when(
            F.col("events") > 0,
            F.col("events_caught") / F.col("events")
        )
    )

    summary = summary.withColumn(
        "lift",
        F.when(
            (F.col("events") > 0) & (F.col("loans") > 0),
            F.col("precision")
            / (F.col("events") / F.col("loans"))
        )
    )

    summary = summary.withColumn("model", F.lit(model_name))
    summary = summary.withColumn("population", F.lit(population))

    return summary.select(
        "model", "population", "loans", "events",
        "alerts", "events_caught", "precision", "recall", "lift"
    )

In [0]:
model_scores = validation_scores.withColumn(
    "ranking_score", F.col("risk_score")
)

model_monthly = monthly_alert_counts(model_scores)

model_summary = summarize_alerts(
    model_monthly,
    "Logistic regression",
    "All eligible loans"
)


# Simple benchmark: rank 60-day late loans first, then 30-day late loans.
rule_scores = validation_scores.withColumn(
    "ranking_score", F.col("delinquency_months").cast("double")
)

rule_monthly = monthly_alert_counts(rule_scores)

rule_summary = summarize_alerts(
    rule_monthly,
    "Current delinquency rule",
    "All eligible loans"
)


# Separate early-warning assessment among loans currently at status 00.
healthy_scores = model_scores.filter(
    F.col("delinquency_months") == 0
)

healthy_monthly = monthly_alert_counts(healthy_scores)

healthy_summary = summarize_alerts(
    healthy_monthly,
    "Logistic regression",
    "Current status 00"
)

comparison = (
    model_summary
    .unionByName(rule_summary)
    .unionByName(healthy_summary)
)

display(comparison)

model,population,loans,events,alerts,events_caught,precision,recall,lift
Logistic regression,All eligible loans,1247263,6134,62368,2622,0.042040790148794256,0.4274535376589501,8.548405941205669
Current delinquency rule,All eligible loans,1247263,6134,62368,2049,0.03285338635197537,0.334039778284969,6.680276038722506
Logistic regression,Current status 00,1239174,4250,61963,822,0.013265981311427788,0.19341176470588237,3.867966853084051


In [0]:
# the model catches 573 more positive loan-months, about 28% more, than the simple rule.

In [0]:
# Save the fitted model
import os

# Create a place to keep the models
spark.sql("""
    CREATE VOLUME IF NOT EXISTS
    workspace.mortgage_risk.model_artifacts
""")

model_path = (
    "/Volumes/workspace/mortgage_risk/model_artifacts/"
    "logistic_regression_v1"
)

# Save the model once
if os.path.exists(model_path):
    print("The baseline model is already saved.")
else:
    baseline_model.write().save(model_path)
    print("Baseline model saved.")

Baseline model saved.


In [0]:
# Save the validation predictions
score_table = "workspace.mortgage_risk.validation_scores_lr"

if spark.catalog.tableExists(score_table):
    print("The validation scores are already saved.")
else:
    validation_scores.write.format("delta").mode("error").saveAsTable(
        score_table
    )
    print("Validation scores saved.")

The validation scores are already saved.
